In [ ]:
import numpy as np
import scipy.io
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.sequence import pad_sequences
from joblib import dump
from tensorflow.keras.models import load_model

#%% Load Data with Temporal Segments
def load_data(subjects, emotion_map):
    X, y, sub_ids = [], [], []
    for subj in subjects:
        data = scipy.io.loadmat(f'EEG_features/{subj}.mat')
        for key in data.keys():
            if key.startswith('de_LDS_'):
                video_id = int(key.split('_')[-1])
                if video_id in emotion_map:
                    # Raw segments: (num_segments, 5 bands, 62 channels)
                    trial = data[key]
                    # Flatten frequency bands and channels
                    trial = trial.reshape(trial.shape[0], -1)  # (segments, 5*62=310)
                    X.append(trial)
                    y.append(emotion_map[video_id])
                    sub_ids.append(subj)
    return X, np.array(y), np.array(sub_ids)

#%% Load Emotion Labels
label_df = pd.read_excel('emotion_label_and_stimuli_order.xlsx', header=None)
emotion_map = {}

for i in range(0, len(label_df), 2):
    current_row = label_df.iloc[i]
    if pd.isna(current_row[0]) or "Video index" not in str(current_row[0]):
        continue
    video_indices = [int(x) for x in current_row[1:22] if not pd.isna(x)]
    emotion_row = label_df.iloc[i + 1]
    emotions = [str(x).strip() for x in emotion_row[1:22] if not pd.isna(x)]
    for vid, emo in zip(video_indices, emotions):
        emotion_map[vid] = emo


# Filter target emotions
target_emotions = ['Happy','Fear','Disgust','Anger']
emotion_map = {vid: emo for vid, emo in emotion_map.items() if emo in target_emotions}
print(emotion_map)
#%% Subject-Independent Split (BEFORE Preprocessing)
subjects = range(1, 21)
X, y, sub_ids = load_data(subjects, emotion_map)
test_subjects = [1, 5, 10, 15, 20]
test_mask = np.isin(sub_ids, test_subjects)
X_train, X_test = [X[i] for i in range(len(X)) if not test_mask[i]], [X[i] for i in range(len(X)) if test_mask[i]]
y_train, y_test = y[~test_mask], y[test_mask]



#%% Padding & Normalization
# Pad sequences to max length in training set
max_length = max(len(seq) for seq in X_train)
X_train_padded = pad_sequences(X_train, maxlen=max_length, padding='post', dtype='float32')
X_test_padded = pad_sequences(X_test, maxlen=max_length, padding='post', dtype='float32')

# Normalize per feature (frequency band × channel)
scaler = StandardScaler()
n_samples, n_timesteps, n_features = X_train_padded.shape
X_train_reshaped = X_train_padded.reshape(-1, n_features)
scaler.fit(X_train_reshaped)
X_train_normalized = scaler.transform(X_train_reshaped).reshape(n_samples, n_timesteps, n_features)
X_test_reshaped = X_test_padded.reshape(-1, n_features)
X_test_normalized = scaler.transform(X_test_reshaped).reshape(X_test_padded.shape)
np.save('X_test_normalized.npy', X_test_normalized)

#%% Label Encoding
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

#%% 1D CNN + LSTM Architecture
model = Sequential([
    Conv1D(128, kernel_size=3, activation='relu', input_shape=(max_length, 310)),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),
    
    Conv1D(256, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),
    
    LSTM(128, return_sequences=True),
    LSTM(64),
    
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

#%% Model Configuration
optimizer = Adam(learning_rate=0.001)
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

#%% Class Weighting & Callbacks
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_encoded), y=y_train_encoded)
class_weights = dict(enumerate(class_weights))

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5)
]

#%% Training
history = model.fit(
    X_train_normalized, y_train_encoded,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=callbacks
)

#%% Evaluation
test_loss, test_acc = model.evaluate(X_test_normalized, y_test_encoded)
y_pred = np.argmax(model.predict(X_test_normalized), axis=1)

print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(classification_report(y_test_encoded, y_pred, target_names=le.classes_))

#%% Visualization
# Training Curve
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Training History')
plt.legend()
plt.show()

# t-SNE Plot (Training Data)
X_tsne = TSNE(n_components=2).fit_transform(X_train_normalized.reshape(-1, max_length*310))
plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y_train_encoded, alpha=0.6)
plt.title("t-SNE of EEG Features")
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test_encoded, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

model.save('emotion_detection_model.h5')  # Save the trained model
dump(scaler, 'scaler.joblib')           
dump(le, 'label_encoder.joblib')
np.save('maximum_length.npy', max_length)